In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, gc
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import joblib
import psutil

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import xgboost as xgb
import lightgbm as lgb

def ram_usage():
    """In mức RAM hiện tại đang dùng."""
    used = psutil.Process().memory_info().rss / 1024**3
    total = psutil.virtual_memory().total / 1024**3
    print(f'  [RAM] {used:.2f} GB / {total:.2f} GB ({used/total*100:.1f}%)')

print('All imports OK')
ram_usage()

All imports OK
  [RAM] 10.58 GB / 52.96 GB (20.0%)


In [ ]:
# --- Dataset config ---
HF_REPO_ID = "PandaLT/forecast-new"
DATASET_SUBFOLDER = "h24_forecast_weather"
# Đường dẫn (bên trong repo) tới folder chứa các file parquet
DATASET_PATH_IN_REPO = os.path.join("24h-forecast", "dataset", DATASET_SUBFOLDER)

# --- Output dirs ---
OUTPUT_BASE = "/content/drive/MyDrive/forecast-model/weather2"
MODELS_DIR = os.path.join(OUTPUT_BASE, "models")
REPORT_DIR = os.path.join(OUTPUT_BASE, "report")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# --- Kiểm tra GPU ---
try:
    import subprocess
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    HAS_GPU = result.returncode == 0
except Exception:
    HAS_GPU = False

TREE_DEVICE = 'cuda' if HAS_GPU else 'cpu'
print(f'Output: {OUTPUT_BASE}')
print(f'Dataset: {HF_REPO_ID} / {DATASET_SUBFOLDER}')
print(f'GPU available: {HAS_GPU} → tree models will use: {TREE_DEVICE}')


Output: /content/drive/MyDrive/forecast-model/weather2
Dataset: PandaLT/forecast-new / h24_forecast_weather
GPU available: True → tree models will use: cuda


In [ ]:
from huggingface_hub import snapshot_download

local_data = snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)
DATA_DIR = os.path.join(local_data, DATASET_PATH_IN_REPO)
print(f'Downloaded to: {local_data}')
print(f'Data folder: {DATA_DIR}')

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Downloaded to: /root/.cache/huggingface/hub/datasets--PandaLT--forecast-new/snapshots/2b5c9e2b50310ff1403eaf80e27580fc3f5be9d3
Data folder: /root/.cache/huggingface/hub/datasets--PandaLT--forecast-new/snapshots/2b5c9e2b50310ff1403eaf80e27580fc3f5be9d3/24h-forecast/dataset/h24_forecast_weather


In [ ]:
train = pl.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
val   = pl.read_parquet(os.path.join(DATA_DIR, "validation.parquet"))
test  = pl.read_parquet(os.path.join(DATA_DIR, "test.parquet"))

print(f'Train:      {train.shape[0]:>10,} rows | {train["timestamp"].min()} → {train["timestamp"].max()}')
print(f'Validation: {val.shape[0]:>10,} rows | {val["timestamp"].min()} → {val["timestamp"].max()}')
print(f'Test:       {test.shape[0]:>10,} rows | {test["timestamp"].min()} → {test["timestamp"].max()}')
print(f'\nFeatures: {train.columns}')
ram_usage()

Train:      17,482,862 rows | 2016-01-08 00:00:00 → 2017-06-30 23:00:00
Validation:  3,087,774 rows | 2017-07-01 00:00:00 → 2017-09-30 23:00:00
Test:        3,024,447 rows | 2017-10-01 00:00:00 → 2017-12-30 23:00:00

Features: ['timestamp', 'building_id', 'consumption', 'site_id', 'primaryspaceusage', 'sqm', 'timezone', 'hour', 'day_of_week', 'month', 'is_weekend', 'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h', 'rolling_std_24h', 'rolling_mean_168h', 'rolling_std_168h', 'future_airTemperature', 'future_dewTemperature', 'future_windDirection', 'future_windSpeed', 'target']
  [RAM] 15.90 GB / 52.96 GB (30.0%)


In [ ]:

USELESS_COLS = {
    "site_id"        # redundant với building_id
}

EXCLUDE_COLS = {"target", "timestamp",} | USELESS_COLS

actually_excluded = [c for c in train.columns if c in EXCLUDE_COLS]
print(f'Excluded ({len(actually_excluded)}): {actually_excluded}')

FEATURE_COLS = [c for c in train.columns if c not in EXCLUDE_COLS]

# Tự động detect categorical cols
CATEGORICAL_COLS = [
    c for c in FEATURE_COLS
    if train[c].dtype in (pl.String, pl.Utf8, pl.Categorical)
]
NUMERIC_COLS = [c for c in FEATURE_COLS if c not in CATEGORICAL_COLS]

# Liệt kê future weather cols đang có
future_wx_cols = [c for c in FEATURE_COLS if c.startswith("future_")]
print(f'\nFuture weather features ({len(future_wx_cols)}): {future_wx_cols}')
print(f'Total features: {len(FEATURE_COLS)}')
print(f'Categorical: {CATEGORICAL_COLS}')

# ── Helper functions ────────────────────────────────────────────────────────

def downcast_floats(df: pl.DataFrame) -> pl.DataFrame:
    cast_exprs = [
        pl.col(c).cast(pl.Float32) if df[c].dtype == pl.Float64 else pl.col(c)
        for c in df.columns
    ]
    return df.select(cast_exprs)


def encode_and_process(train_df, val_df, test_df, cat_cols, feature_cols):
    needed = feature_cols + ["target"]

    vocab = {}
    for col in cat_cols:
        unique_vals = sorted(
            str(v) for v in train_df[col].drop_nulls().unique().to_list()
        )
        vocab[col] = {v: idx for idx, v in enumerate(unique_vals)}

    def encode_df(df):
        df = df.select(needed)
        encode_exprs = []
        for col in df.columns:
            if col in vocab:
                mapping = vocab[col]
                encode_exprs.append(
                    pl.col(col)
                    .cast(pl.String)
                    .map_elements(lambda x, m=mapping: m.get(x, -1), return_dtype=pl.Int32)
                    .cast(pl.Float32)
                    .alias(col)
                )
            else:
                encode_exprs.append(pl.col(col))
        df = df.select(encode_exprs)
        return downcast_floats(df)

    return encode_df(train_df), encode_df(val_df), encode_df(test_df)


# ── Null check — pipeline đảm bảo 0 null, nhưng verify để chắc chắn ─────────
for split_name, split_df in [("train", train), ("val", val), ("test", test)]:
    null_counts = split_df.select([
        pl.col(c).is_null().sum().alias(c) for c in split_df.columns
    ])
    total_nulls = sum(null_counts.row(0))
    if total_nulls > 0:
        bad = {c: null_counts[c][0] for c in split_df.columns if null_counts[c][0] > 0}
        raise RuntimeError(f"[{split_name}] Dataset có null — cần re-run pipeline: {bad}")
print("Null check passed — 0 nulls in all splits.")

# ── Load → Encode → Free ────────────────────────────────────────────────────
needed_cols = FEATURE_COLS + ["target"]

train_slim, val_slim, test_slim = encode_and_process(
    train.select(needed_cols),
    val.select(needed_cols),
    test.select(needed_cols),
    CATEGORICAL_COLS,
    FEATURE_COLS,
)

del train, val, test
gc.collect()
print('Freed original dataframes')
ram_usage()

# ── Extract numpy ────────────────────────────────────────────────────────────
remaining_str = [c for c in FEATURE_COLS if train_slim[c].dtype == pl.String]
if remaining_str:
    raise RuntimeError(f"Vẫn còn string columns chưa encode: {remaining_str}")

X_train = train_slim.select(FEATURE_COLS).to_numpy(allow_copy=True).astype(np.float32)
y_train = train_slim["target"].to_numpy()

X_val   = val_slim.select(FEATURE_COLS).to_numpy(allow_copy=True).astype(np.float32)
y_val   = val_slim["target"].to_numpy()

X_test  = test_slim.select(FEATURE_COLS).to_numpy(allow_copy=True).astype(np.float32)
y_test  = test_slim["target"].to_numpy()

del train_slim, val_slim, test_slim
gc.collect()

print(f'X_train: {X_train.shape}, dtype={X_train.dtype}  ({X_train.nbytes/1024**3:.2f} GB)')
print(f'X_val:   {X_val.shape}')
print(f'X_test:  {X_test.shape}')
ram_usage()


Excluded (4): ['timestamp', 'consumption', 'site_id', 'target']

Future weather features (4): ['future_airTemperature', 'future_dewTemperature', 'future_windDirection', 'future_windSpeed']
Total features: 19
Categorical: ['building_id', 'primaryspaceusage', 'timezone']
Null check passed — 0 nulls in all splits.
Freed original dataframes
  [RAM] 15.91 GB / 52.96 GB (30.0%)
X_train: (17482862, 19), dtype=float32  (1.24 GB)
X_val:   (3087774, 19)
X_test:  (3024447, 19)
  [RAM] 15.89 GB / 52.96 GB (30.0%)


**Check NaN/Null Before Training**

In [ ]:

from sklearn.impute import SimpleImputer

# Kiểm tra mức độ NaN trước
nan_counts = np.isnan(X_train).sum(axis=0)
nan_cols = [(FEATURE_COLS[i], int(nan_counts[i])) for i in np.where(nan_counts > 0)[0]]
print(f'Columns with NaN: {len(nan_cols)} / {len(FEATURE_COLS)}')
for col, cnt in nan_cols[:10]:  # chỉ in 10 đầu
    print(f'  {col}: {cnt:,} NaN ({cnt/len(X_train)*100:.1f}%)')

# Dùng median — robust hơn mean với outliers, phù hợp energy data
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train).astype(np.float32)
X_val   = imputer.transform(X_val).astype(np.float32)
X_test  = imputer.transform(X_test).astype(np.float32)

# Lưu imputer để dùng lại lúc inference
joblib.dump(imputer, os.path.join(MODELS_DIR, 'imputer.pkl'))

print(f'\nAfter imputation — NaN còn lại: {np.isnan(X_train).sum()}')
ram_usage()

Columns with NaN: 0 / 19

After imputation — NaN còn lại: 0
  [RAM] 15.89 GB / 52.96 GB (30.0%)


**Training**

In [ ]:
xgb_params = dict(
    objective="reg:absoluteerror",

    n_estimators=2000,
    early_stopping_rounds=200,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1 if TREE_DEVICE == 'cpu' else 1,
    tree_method='hist',
    device=TREE_DEVICE,
)


models = {
    "Linear Regression": LinearRegression(n_jobs=-1),
    "XGBoost": xgb.XGBRegressor(**xgb_params)
}

trained = {}

for name, model in models.items():
    print(f'\n[{name}] Training...')
    ram_usage()

    if name in ("XGBoost", "LightGBM"):
        if name == "LightGBM":
            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                callbacks=[
                    lgb.early_stopping(100, verbose=False),
                    lgb.log_evaluation(period=100),
                ],
            )
        else:
            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                verbose=100,
            )
    else:
        model.fit(X_train, y_train)

    trained[name] = model
    gc.collect()
    print(f'  ✓ Done: {name}')
    ram_usage()



[Linear Regression] Training...
  [RAM] 15.89 GB / 52.96 GB (30.0%)
  ✓ Done: Linear Regression
  [RAM] 15.89 GB / 52.96 GB (30.0%)

[XGBoost] Training...
  [RAM] 15.89 GB / 52.96 GB (30.0%)
[0]	validation_0-mae:120.39782
[100]	validation_0-mae:16.15331
[200]	validation_0-mae:14.54515
[300]	validation_0-mae:14.24099
[400]	validation_0-mae:13.82408
[500]	validation_0-mae:13.42301
[600]	validation_0-mae:13.23365
[700]	validation_0-mae:13.06207
[800]	validation_0-mae:12.96191
[900]	validation_0-mae:12.83947
[1000]	validation_0-mae:12.72737
[1100]	validation_0-mae:12.65341
[1200]	validation_0-mae:12.57624
[1300]	validation_0-mae:12.50182
[1400]	validation_0-mae:12.43023
[1500]	validation_0-mae:12.37782
[1600]	validation_0-mae:12.32209
[1700]	validation_0-mae:12.27991
[1800]	validation_0-mae:12.24230
[1900]	validation_0-mae:12.21954
[1999]	validation_0-mae:12.19741
  ✓ Done: XGBoost
  [RAM] 15.95 GB / 52.96 GB (30.1%)


**Evaluate**

In [ ]:
def calculate_metrics(y_true, y_pred):
    """Calculate MAE, RMSE, MAPE, SMAPE. MAPE dùng mask y_true > 1 để tránh nổ."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    # MAPE chỉ tính trên target > 1 để tránh chia cho 0 hoặc giá trị gần 0 (review #3)
    mask = y_true > 1
    if mask.sum() > 0:
        mape = mean_absolute_percentage_error(y_true[mask], y_pred[mask]) * 100
    else:
        mape = float('nan')
    smape = (
        2 * np.mean(np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100
    )
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "SMAPE": smape}


rows = []
for name, model in trained.items():
    y_val_pred  = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    val_metrics  = calculate_metrics(y_val, y_val_pred)
    test_metrics = calculate_metrics(y_test, y_test_pred)

    row = {"Model": name}
    for k, v in val_metrics.items():
        row[f"Val_{k}"] = round(v, 4)
    for k, v in test_metrics.items():
        row[f"Test_{k}"] = round(v, 4)
    rows.append(row)

leaderboard = pl.DataFrame(rows)
print(leaderboard)

**Ranking most important feature**

In [ ]:
best_name  = leaderboard.sort("Test_MAPE")["Model"][0]
best_model = trained[best_name]
y_pred     = best_model.predict(X_test)

tree_model = best_model

if hasattr(tree_model, "get_booster"):
    # XGBoost: dùng gain thay vì weight (default của feature_importances_)
    score = tree_model.get_booster().get_score(importance_type="gain")
    # Normalize về [0,1]
    total = sum(score.values()) or 1
    importances = np.array([
        score.get(f"f{i}", 0) / total for i in range(len(FEATURE_COLS))
    ])
elif hasattr(tree_model, "feature_importances_"):
    # LightGBM, RandomForest
    importances = tree_model.feature_importances_
else:
    importances = None

if importances is not None:
    top_k = 20
    idx = np.argsort(importances)[-top_k:]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh([FEATURE_COLS[i] for i in idx], importances[idx])
    ax.set_xlabel("Importance (gain)")
    ax.set_title(f"Top {top_k} Features — {best_name}")
    plt.tight_layout()
    fi_path = os.path.join(REPORT_DIR, "feature_importance.png")
    plt.savefig(fi_path, dpi=120)
    plt.show()
    print(f"Saved: {fi_path}")
else:
    print(f"{best_name} does not expose feature importances.")

**Evaluate Per Building-id MAE & MAPE**

In [ ]:
_schema = pl.read_parquet_schema(os.path.join(DATA_DIR, "test.parquet"))
BID_COL = next(
    (c for c in _schema
     if c.lower().replace("-", "_") in ("building_id", "buildingid", "building")),
    None,
)
if BID_COL is None:
    raise RuntimeError(f"Không tìm thấy cột building_id. Các cột có: {list(_schema)}")

building_ids = pl.read_parquet(
    os.path.join(DATA_DIR, "test.parquet"), columns=[BID_COL]
)[BID_COL].to_numpy()
assert len(building_ids) == len(y_test), \
    f"Lệch số hàng: {len(building_ids)} ids vs {len(y_test)} test rows"

# Dự đoán của best model (đã chọn ở cell feature-importance)
y_test_pred = best_model.predict(X_test)

# Gom theo building → MAE & MAPE từng building
per_b = (
    pl.DataFrame({
        "building_id": building_ids,
        "y_true": y_test,
        "y_pred": y_test_pred,
    })
    .with_columns((pl.col("y_true") - pl.col("y_pred")).abs().alias("abs_err"))
    .group_by("building_id")
    .agg([
        pl.col("abs_err").mean().alias("MAE"),
        # MAPE chỉ trên y_true > 1 để tránh chia cho ~0 (đồng bộ calculate_metrics)
        (
            (pl.col("abs_err").filter(pl.col("y_true") > 1)
             / pl.col("y_true").filter(pl.col("y_true") > 1)).mean() * 100
        ).alias("MAPE"),
        pl.len().alias("n_samples"),
    ])
    .sort("MAE", descending=True)
)

mae_mean,  mae_median  = per_b["MAE"].mean(),  per_b["MAE"].median()
mape_mean, mape_median = per_b["MAPE"].mean(), per_b["MAPE"].median()

print(f"Per-building — {best_name}  ({per_b.height} buildings)")
print(f"  MAE : mean={mae_mean:.4f} | median={mae_median:.4f}")
print(f"  MAPE: mean={mape_mean:.4f}% | median={mape_median:.4f}%")
print("\nTop 10 building MAE cao nhất:")
print(per_b.head(10))

# ── Phân phối MAE / MAPE theo building ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(per_b["MAE"].to_numpy(), bins=50, color="#4C72B0", edgecolor="white")
axes[0].axvline(mae_mean,   color="red",   ls="--", label=f"mean={mae_mean:.2f}")
axes[0].axvline(mae_median, color="green", ls="--", label=f"median={mae_median:.2f}")
axes[0].set_title("Distribution of MAE per building")
axes[0].set_xlabel("MAE"); axes[0].set_ylabel("# buildings"); axes[0].legend()

axes[1].hist(per_b["MAPE"].drop_nulls().to_numpy(), bins=50, color="#DD8452", edgecolor="white")
axes[1].axvline(mape_mean,   color="red",   ls="--", label=f"mean={mape_mean:.2f}%")
axes[1].axvline(mape_median, color="green", ls="--", label=f"median={mape_median:.2f}%")
axes[1].set_title("Distribution of MAPE per building")
axes[1].set_xlabel("MAPE (%)"); axes[1].set_ylabel("# buildings"); axes[1].legend()
plt.tight_layout()
dist_path = os.path.join(REPORT_DIR, "per_building_error_dist.png")
plt.savefig(dist_path, dpi=120); plt.show()
print(f"Saved: {dist_path}")

# ── Lưu MAE per building (đã sort) ra CSV để kiểm tra sau ───────────────────
mae_csv = os.path.join(REPORT_DIR, "mae_per_building.csv")
per_b.write_csv(mae_csv)
print(f"Saved: {mae_csv}")


**Mean Consumption Per Building-id**

In [ ]:
# ── Mức tiêu thụ trung bình (target) mỗi building trên test set ─────────────
mean_consumption = (
    pl.DataFrame({"building_id": building_ids, "consumption": y_test})
    .group_by("building_id")
    .agg([
        pl.col("consumption").mean().alias("mean_consumption"),
        pl.col("consumption").median().alias("median_consumption"),
        pl.col("consumption").std().alias("std_consumption"),
        pl.col("consumption").min().alias("min_consumption"),
        pl.col("consumption").max().alias("max_consumption"),
        pl.len().alias("n_samples"),
    ])
    .sort("mean_consumption", descending=True)
)

print(f"Mean consumption per building ({mean_consumption.height} buildings)")
print(mean_consumption.head(10))

# Ghép với MAE để xem quan hệ giữa độ lớn tiêu thụ và sai số
joined = mean_consumption.join(
    per_b.select(["building_id", "MAE", "MAPE"]), on="building_id", how="left"
)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(range(mean_consumption.height),
          mean_consumption["mean_consumption"].to_numpy(), color="#55A868")
ax[0].set_title("Mean consumption per building (sorted desc)")
ax[0].set_xlabel("building rank"); ax[0].set_ylabel("mean consumption")

ax[1].scatter(joined["mean_consumption"].to_numpy(), joined["MAE"].to_numpy(),
              s=14, alpha=0.5, color="#C44E52")
ax[1].set_title("MAE vs mean consumption")
ax[1].set_xlabel("mean consumption"); ax[1].set_ylabel("MAE")
plt.tight_layout()
mc_path = os.path.join(REPORT_DIR, "mean_consumption_per_building.png")
plt.savefig(mc_path, dpi=120); plt.show()
print(f"Saved: {mc_path}")

mc_csv = os.path.join(REPORT_DIR, "mean_consumption_per_building.csv")
mean_consumption.write_csv(mc_csv)
print(f"Saved: {mc_csv}")


**Remove High-MAE-loss building and Recompute the average MAE**

In [ ]:
# ── Loại building MAE cao bất thường rồi tính lại MAE & MAPE trung bình ─────
# Ngưỡng theo IQR (robust với outlier): MAE > Q3 + 1.5 * IQR
q1, q3 = per_b["MAE"].quantile(0.25), per_b["MAE"].quantile(0.75)
iqr = q3 - q1
threshold = q3 + 1.5 * iqr
print(f"Ngưỡng MAE (Q3 + 1.5*IQR) = {threshold:.4f}")

high_mae = per_b.filter(pl.col("MAE") > threshold).sort("MAE", descending=True)
kept     = per_b.filter(pl.col("MAE") <= threshold)
print(f"\nBuilding bị loại (MAE cao): {high_mae.height} / {per_b.height}")
print(high_mae.head(10))

# Mask các sample thuộc building được giữ lại
kept_ids = set(kept["building_id"].to_list())
mask = np.array([b in kept_ids for b in building_ids])

# ── Macro = trung bình theo từng building ───────────────────────────────────
macro_mae_before,  macro_mae_after  = per_b["MAE"].mean(),  kept["MAE"].mean()
macro_mape_before, macro_mape_after = per_b["MAPE"].mean(), kept["MAPE"].mean()  # .mean() bỏ qua null

# ── Micro = tính trực tiếp trên toàn bộ sample ─────────────────────────────
def _mape(yt, yp):
    m = yt > 1  # đồng bộ với calculate_metrics: chỉ tính trên target > 1
    return mean_absolute_percentage_error(yt[m], yp[m]) * 100 if m.sum() else float("nan")

micro_mae_before,  micro_mae_after  = mean_absolute_error(y_test, y_test_pred), mean_absolute_error(y_test[mask], y_test_pred[mask])
micro_mape_before, micro_mape_after = _mape(y_test, y_test_pred),               _mape(y_test[mask], y_test_pred[mask])

def _pct(b, a):
    return f"{(a - b) / b * 100:+.1f}%" if b else "n/a"

print("\n── So sánh trước → sau khi loại building MAE cao ──")
print(f"  Macro MAE : {macro_mae_before:.4f}  →  {macro_mae_after:.4f}   ({_pct(macro_mae_before, macro_mae_after)})")
print(f"  Macro MAPE: {macro_mape_before:.4f}% →  {macro_mape_after:.4f}%  ({_pct(macro_mape_before, macro_mape_after)})")
print(f"  Micro MAE : {micro_mae_before:.4f}  →  {micro_mae_after:.4f}   ({_pct(micro_mae_before, micro_mae_after)})")
print(f"  Micro MAPE: {micro_mape_before:.4f}% →  {micro_mape_after:.4f}%  ({_pct(micro_mape_before, micro_mape_after)})")
print(f"  Sample giữ lại: {mask.sum():,} / {len(mask):,} ({mask.mean()*100:.1f}%)")

high_csv = os.path.join(REPORT_DIR, "high_mae_buildings.csv")
high_mae.write_csv(high_csv)
print(f"\nSaved: {high_csv}")

In [ ]:
print(train.select("target").describe())


shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ target      │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 1.7482862e7 │
│ null_count ┆ 0.0         │
│ mean       ┆ 146.136782  │
│ std        ┆ 259.650162  │
│ min        ┆ 0.0001      │
│ 25%        ┆ 20.1        │
│ 50%        ┆ 60.425      │
│ 75%        ┆ 157.6742    │
│ max        ┆ 8767.8      │
└────────────┴─────────────┘
